# Step 4. Initial conditions: ERA5 + CLIM_BASE fields + DART perturbations (HYB_CLIMFILL_DARTPERT20)

## Purpose

Builds a third 20-member initial ensemble: the same ERA5 atmosphere, DART perturbations and HYB land, river and sea-ice restarts as step 3 (HYB_DARTPERT20C), but every field ERA5 does not provide (ozone, Linoz gases, chemistry, aerosols, hydrometeor numbers, surface fields) from the free-running CLIM_BASE (`FREE_CLIM`) state of the same date instead of the DART mean. The difference from step 3 is only the source of those fields, so comparing the two isolates how much they matter. Against the DART mean (16 May 2012) the CLIM_BASE ozone column is about the same (279 vs 280 DU), but black carbon is 1.40, dust 1.31 and cloud droplet number 1.26 times larger, so this mainly varies aerosols and hydrometeor numbers.

The ERA5 initial files `DART_INIT/ATM/v3.LR.ne30pg2.ERA5.eam.i.<date>.nc` hold only T, U, V, Q, PS, CLDLIQ and CLDICE. EAM starts every missing field from zero, and the radiation reads its gases from tracers (`rad_climate = 'A:H2OLNZ:H2O', 'A:O3:O3', 'A:N2OLNZ:N2O', 'A:CH4LNZ:CH4'`), so a run from the bare ERA5 file starts radiatively dry with no ozone, N2O or CH4 (day-1 land TREFHT about 6 K too cold in the first HYB_DARTPERT20 build). The ERA5 states are therefore completed first.

## Workflow

```text
step 1  01_select_dart_20_member_subset.ipynb   20 DART members per season
          |
          +--> step 2  02_ic_free_clim_dartpert.ipynb      CLIM_BASE state          + DART perturbations -> FREECLIM_DARTPERT20
          +--> step 3  03_ic_era5_dartfill_dartpert.ipynb  ERA5 + DART-mean fields  + DART perturbations -> HYB_DARTPERT20C
          +--> step 4  04_ic_era5_climfill_dartpert.ipynb  ERA5 + CLIM_BASE fields  + DART perturbations -> HYB_CLIMFILL_DARTPERT20
```

This notebook runs two steps:

* **Step 4a** completes the ERA5 states: ERA5 atmosphere, every other field from the free-running CLIM_BASE (FREE_CLIM) initial file of the same date (`ATM_COMPLETE_CLIMFILL/v3.LR.ne30pg2.ERA5climfill.eam.i.<date>.nc`);
* **Step 4b** adds the DART perturbations to the completed states (with the same DATM-forced HYB land, river and sea-ice restarts as step 3).

## Method

### Step 4a. Completion

For each start date `util/complete_era5_ic.py`:

1. takes every field of a full EAM initial file from the fill source (the free-running CLIM_BASE (FREE_CLIM) initial file of the same date), in the native layout (dynamics fields on `ncol_d`, physics-grid surface fields on `ncol`);
2. replaces T, U, V, Q, PS, CLDLIQ and CLDICE with the ERA5 values (same GLL columns and hybrid levels, checked);
3. sets the radiative water vapour to the ERA5 moisture with the fill source's relative difference from Q: `H2OLNZ = Q_ERA5 * (H2OLNZ_fill / Q_fill)`, the ratio bounded to `H2O_RATIO_BOUNDS` (multiplicative, so it stays positive where ERA5 is drier).

Checks: the ERA5 fields are exact, no field of the fill file is missing, nothing is non-finite, the total ozone column is realistic, and `H2OLNZ/Q` is near 1 below 500 hPa.

### Step 4b. Perturbation

For the 20 selected DART members $S$ of the season (step 1) and member $s_i \in S$:

$$\delta A_{s_i} = A_{\mathrm{DART},s_i} - \overline{A}_{\mathrm{DART},S}, \qquad
A_{\mathrm{EN}(i+1)} = A_{\mathrm{ref}} + \alpha\,\delta A_{s_i},\ i = 1,\dots,19, \qquad A_{\mathrm{EN01}} = A_{\mathrm{ref}}.$$

The 20th selected member is in the mean but gives no output member, so the output ensemble mean is offset from the reference by $-\alpha\,\delta A_{\mathrm{omitted}}/20$; the written files are checked against that offset (not enforced for the clipped moisture fields). Only the perturbed fields change: the structure, attributes and every other field of the reference file are checked to be identical. The reference's land, river, sea-ice, coupler restarts and rpointers are linked into each date directory.

T, U, V, Q, PS and H2OLNZ are perturbed (H2OLNZ with Q, so the radiation sees each member's moisture); Q and H2OLNZ are clipped at 0.

## Outputs

Each date directory `<OUTPUT_ROOT>/<date>/` holds `<OUT_PREFIX>.EN01-EN20.eam.i.<date>.nc`, the component restart links, `output_to_dart_member_mapping.csv`, `perturbation_diagnostics.csv`, `q_clipping_diagnostics.csv`, `written_file_validation.csv` and `dart_perturbation_metadata.json`; `<OUTPUT_ROOT>/all_*.csv` summarize all dates.

## 1. Libraries

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

## 2. User setup

Plain values only, so `papermill -p` overrides work.

In [2]:
DART_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/e3sm_dart")   # <case>/[EN##/]archive/rest/<date>/
SELECTION_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart_analysis/ensemble_ic/dart_subset_selection")  # checked when present
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
SEASONS = None            # None: both; or "dec2011" / "may2012"
OVERWRITE = False         # True rewrites existing outputs; False validates and reuses them

ERA5_ATM_DIR = Path("/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM")       # bare ERA5 states
ERA5_PATTERN = "v3.LR.ne30pg2.ERA5.eam.i.{date}.nc"
COMPLETE_DIR = Path("/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE_CLIMFILL")
COMPLETE_PATTERN = "v3.LR.ne30pg2.ERA5climfill.eam.i.{date}.nc"
H2O_RATIO_BOUNDS = (0.2, 5.0)
CLIM_CASE = "F20TR_ne30pg2_r05_IcoswISC30E3r5.FREE_CLIM"
CLIM_REST_ROOT = DART_ROOT / CLIM_CASE / "archive" / "rest"                 # <date>/<case>.eam.i.<date>.nc
HYB_REST_ROOT = DART_ROOT / "20241109.v3-LR.DATM.ne30pg2_r05_IcoswISC30E3r5.pm-cpu" / "archive" / "rest"
OUTPUT_ROOT = DART_ROOT / "hyb_climfill_dartpert20_atm_multidate"
OUT_PREFIX = "HYB_CLIMFILL_DARTPERT20"
PERTURB_VARS = ["T", "U", "V", "Q", "PS", "H2OLNZ"]

## 3. Modules and inputs

In [3]:
pkg = WORK_DIR.resolve() / "ensemble_perturbations"
if str(pkg) not in sys.path:
    sys.path.insert(0, str(pkg))
from util import dart_perturbations as dp
from util.complete_era5_ic import complete_era5_ic, validate


def fill_files(season, date):
    """The fill source of one date: the CLIM_BASE (FREE_CLIM) initial file."""
    return [CLIM_REST_ROOT / date / f"{CLIM_CASE}.eam.i.{date}.nc"]

FILL_LABEL = "free-running CLIM_BASE (FREE_CLIM) state"
seasons = dp.dart_date_sets(DART_ROOT, SEASONS, SELECTION_ROOT)
settings = dp.Settings(variables=tuple(PERTURB_VARS), overwrite=OVERWRITE)
for s in seasons:
    print(f"[SEASON] {s.name}: {s.case}, dates {s.dates}, members {s.members}")
reference = dp.ReferenceCase(
    name="HYB_CLIMFILL",
    atm_file=str(COMPLETE_DIR / COMPLETE_PATTERN),
    rest_dir=str(HYB_REST_ROOT / "{date}"),           # DATM-forced land, river, sea ice and coupler restarts
    output_root=OUTPUT_ROOT, out_prefix=OUT_PREFIX)

[SEASON] dec2011: DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy, dates ['2011-12-16-00000', '2011-12-21-00000', '2011-12-26-00000', '2011-12-31-00000'], members [1, 3, 4, 7, 8, 9, 10, 14, 17, 18, 20, 21, 25, 26, 28, 31, 35, 36, 37, 40]
[SEASON] may2012: DARTEN40S1_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy, dates ['2012-05-16-00000', '2012-05-21-00000', '2012-05-26-00000', '2012-05-31-00000'], members [2, 3, 5, 7, 11, 12, 13, 14, 15, 16, 20, 22, 24, 26, 27, 28, 31, 32, 34, 38]


## 4. Step 4a. Complete the ERA5 states

Existing completed files are only re-checked unless `OVERWRITE`.

In [4]:
rows = []
for s in seasons:
    for d in s.dates:
        era5, out = ERA5_ATM_DIR / ERA5_PATTERN.format(date=d), COMPLETE_DIR / COMPLETE_PATTERN.format(date=d)
        fill = fill_files(s, d)
        if out.exists() and not OVERWRITE:
            r = validate(out, era5, fill[0])
        else:
            r = complete_era5_ic(era5, fill, out, ratio_bounds=H2O_RATIO_BOUNDS, fill_label=FILL_LABEL, overwrite=OVERWRITE)
        rows.append(dict(season=s.name, date=d, **{k: v for k, v in r.items() if k != "file"}))
        print(f"[COMPLETE] {out}")
completion = pd.DataFrame(rows)
completion.to_csv(COMPLETE_DIR / "completion_checks.csv", index=False)
display(completion)
bad = completion[(completion.missing_vs_dart.str.len() > 0) | (completion.nonfinite_vars.str.len() > 0)
                 | (completion.era5_max_abs_diff.apply(lambda x: max(x.values())) > 0)
                 | ~completion.toz_du_mean.between(200, 400)]
if not bad.empty:
    raise RuntimeError(f"completed files failing the checks:\n{bad[['season', 'date']]}")
print("[CHECK] all completed files pass")

[COMPLETE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE_CLIMFILL/v3.LR.ne30pg2.ERA5climfill.eam.i.2011-12-16-00000.nc
[COMPLETE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE_CLIMFILL/v3.LR.ne30pg2.ERA5climfill.eam.i.2011-12-21-00000.nc
[COMPLETE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE_CLIMFILL/v3.LR.ne30pg2.ERA5climfill.eam.i.2011-12-26-00000.nc
[COMPLETE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE_CLIMFILL/v3.LR.ne30pg2.ERA5climfill.eam.i.2011-12-31-00000.nc
[COMPLETE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE_CLIMFILL/v3.LR.ne30pg2.ERA5climfill.eam.i.2012-05-16-00000.nc
[COMPLETE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE_CLIMFILL/v3.LR.ne30pg2.ERA5climfill.eam.i.2012-05-21-00000.nc
[COMPLETE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE_CLIMFILL/v3.LR.ne30pg2.ERA5climfill.eam.i.2012-05-26-00000.nc
[COMPLETE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE_CLIMFILL/v3.LR.ne30pg2.ERA5climfill.ea

,season,date,missing_vs_dart,era5_max_abs_diff,nonfinite_vars,toz_du_mean,toz_du_min,toz_du_max,h2olnz_over_q_below_500hPa_median,n_vars
0,dec2011,2011-12-16-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],268.397163,208.702938,358.987697,0.993851,129
1,dec2011,2011-12-21-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],268.409184,203.093058,364.579957,0.993787,129
2,dec2011,2011-12-26-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],268.493543,216.244620,372.772802,0.993885,129
3,dec2011,2011-12-31-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],268.804584,225.678427,369.144236,0.993914,129
4,may2012,2012-05-16-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],278.573897,226.123436,391.729346,0.993494,129
5,may2012,2012-05-21-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],278.290879,230.023533,391.112537,0.993356,129
6,may2012,2012-05-26-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],277.760933,218.368724,397.999984,0.993211,129
7,may2012,2012-05-31-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],277.423530,217.439600,390.598972,0.993191,129


[CHECK] all completed files pass


## 5. Step 4b-1. Configure the start dates

One row per start date; every selected DART member is checked against the reference grid and layout.

In [5]:
configs = dp.build_run_configs(reference, seasons, settings)
for c in configs:
    dp.preflight(c, settings)
display(dp.config_table(configs))

,reference,season,date,dart_source,selected,omitted,ref_atm_file,output_dir
0,HYB_CLIMFILL,dec2011,2011-12-16-00000,rest_eam_i,"[1, 3, 4, 7, 8, 9, 10, 14, 17, 18, 20, 21, 25,...",40,/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_...,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
1,HYB_CLIMFILL,dec2011,2011-12-21-00000,rest_eam_i,"[1, 3, 4, 7, 8, 9, 10, 14, 17, 18, 20, 21, 25,...",40,/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_...,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
2,HYB_CLIMFILL,dec2011,2011-12-26-00000,rest_eam_i,"[1, 3, 4, 7, 8, 9, 10, 14, 17, 18, 20, 21, 25,...",40,/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_...,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
3,HYB_CLIMFILL,dec2011,2011-12-31-00000,rest_eam_i,"[1, 3, 4, 7, 8, 9, 10, 14, 17, 18, 20, 21, 25,...",40,/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_...,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
4,HYB_CLIMFILL,may2012,2012-05-16-00000,rest_eam_i,"[2, 3, 5, 7, 11, 12, 13, 14, 15, 16, 20, 22, 2...",38,/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_...,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
5,HYB_CLIMFILL,may2012,2012-05-21-00000,rest_eam_i,"[2, 3, 5, 7, 11, 12, 13, 14, 15, 16, 20, 22, 2...",38,/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_...,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
6,HYB_CLIMFILL,may2012,2012-05-26-00000,rest_eam_i,"[2, 3, 5, 7, 11, 12, 13, 14, 15, 16, 20, 22, 2...",38,/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_...,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...
7,HYB_CLIMFILL,may2012,2012-05-31-00000,rest_eam_i,"[2, 3, 5, 7, 11, 12, 13, 14, 15, 16, 20, 22, 2...",38,/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_...,/global/cfs/cdirs/m4849/zhan391/e3sm_dart/hyb_...


## 6. Step 4b-2. Build the perturbed ensemble

EN01 is a checksum-verified copy of the completed ERA5 state; EN02-EN20 are perturbed. Existing outputs are validated and reused unless `OVERWRITE`.

In [6]:
tables = dp.build_library(configs, settings, display=display)
display(tables["diagnostics"].pivot_table(index="date", columns="variable", values="mean_member_rms_perturbation"))


[DATE] HYB_CLIMFILL dec2011 2011-12-16-00000 (rest_eam_i)
[REUSE] HYB_CLIMFILL_DARTPERT20.EN01.eam.i.2011-12-16-00000.nc (checksum verified)
[WRITE] EN02 <- DART EN01
[REUSE] HYB_CLIMFILL_DARTPERT20.EN02.eam.i.2011-12-16-00000.nc (structure verified)
[WRITE] EN03 <- DART EN03
[REUSE] HYB_CLIMFILL_DARTPERT20.EN03.eam.i.2011-12-16-00000.nc (structure verified)
[WRITE] EN04 <- DART EN04
[REUSE] HYB_CLIMFILL_DARTPERT20.EN04.eam.i.2011-12-16-00000.nc (structure verified)
[WRITE] EN05 <- DART EN07
[REUSE] HYB_CLIMFILL_DARTPERT20.EN05.eam.i.2011-12-16-00000.nc (structure verified)
[WRITE] EN06 <- DART EN08
[REUSE] HYB_CLIMFILL_DARTPERT20.EN06.eam.i.2011-12-16-00000.nc (structure verified)
[WRITE] EN07 <- DART EN09
[REUSE] HYB_CLIMFILL_DARTPERT20.EN07.eam.i.2011-12-16-00000.nc (structure verified)
[WRITE] EN08 <- DART EN10
[REUSE] HYB_CLIMFILL_DARTPERT20.EN08.eam.i.2011-12-16-00000.nc (structure verified)
[WRITE] EN09 <- DART EN14
[REUSE] HYB_CLIMFILL_DARTPERT20.EN09.eam.i.2011-12-16-00000.nc

variable,H2OLNZ,PS,Q,T,U,V
date,,,,,,
2011-12-16-00000,0.000334,150.800345,0.000323,1.423773,3.816014,3.765881
2011-12-21-00000,0.000363,202.720433,0.000353,1.579652,4.319897,4.132760
2011-12-26-00000,0.000342,231.024417,0.000334,1.739338,4.925186,4.618430
2011-12-31-00000,0.000355,231.369799,0.000349,1.927456,5.329812,4.959590
2012-05-16-00000,0.000358,163.616416,0.000359,1.373788,3.650841,3.615675
2012-05-21-00000,0.000360,189.617309,0.000359,1.503054,4.161901,3.949844
2012-05-26-00000,0.000359,231.789512,0.000344,1.588802,4.503715,4.207433
2012-05-31-00000,0.000368,283.704713,0.000363,1.685735,4.759513,4.454924


## 7. Verdict

In [7]:
v = tables["validation"]
clip = tables["q_clipping"]
print(f"[CHECK] {len(configs)} dates, {len(configs) * len(settings.all_members)} member files")
print("[CHECK] ensemble-mean closure:", "pass" if v["passes_tolerance"].all() else "FAIL")
if not clip.empty:
    print(f"[CHECK] max fraction of clipped moisture points: {clip['fraction_q_clipped'].max():.2e}")

[CHECK] 8 dates, 160 member files
[CHECK] ensemble-mean closure: pass
[CHECK] max fraction of clipped moisture points: 2.69e-02
